# Bot Workflow Runs Summary Extractor

**Date:** 2026-02-08  
**Purpose:** Extract key information from GitHub Actions workflow runs and create Excel summary  
**Version:** 007b - Fixed pagination handling and Unicode decoding

**Project Documentation:** `conversation and context docs/Extract Run Info from GitHub Actions Implementation Plan 02-08-2026.md`

This notebook:
1. Fetches workflow runs from GitHub Actions API (handles pagination correctly)
2. Downloads logs for each run (handles Unicode properly with byte-based decoding)
3. Extracts: run number, timestamp, question presence, question number, forecast value, error flag
4. Outputs to Excel file: `products/Runs and Question Numbers.xlsx`

**Fixes in 007b:**
- ✅ ANSI color code stripping
- ✅ Proper pagination handling (13 pages, 1224 runs)
- ✅ Robust UTF-8 encoding (byte-based decode with fallback to latin-1)

---

## Setup & Configuration

In [1]:
# Install openpyxl if needed
# !pip install openpyxl

In [2]:
import json
import re
import subprocess
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict

from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font, Alignment

print("✅ Imports successful")

✅ Imports successful


In [3]:
# Configuration
REPO = "D-Enns/metac-bot-template"
WORKFLOW = "dre_run_bot_on_tournament.yaml"
OUTPUT_FILE = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products/Runs and Question Numbers.xlsx")

# For testing - limit number of runs to process
TEST_LIMIT = 10  # Set to None to process all runs

print(f"Repository: {REPO}")
print(f"Workflow: {WORKFLOW}")
print(f"Output: {OUTPUT_FILE}")
print(f"Test limit: {TEST_LIMIT}")

Repository: D-Enns/metac-bot-template
Workflow: dre_run_bot_on_tournament.yaml
Output: C:\Users\Donni\projects\metac_bot_Spring_2026\products\Runs and Question Numbers.xlsx
Test limit: 10


In [4]:
# Data structure for extracted run information
@dataclass
class RunInfo:
    workflow_run_number: int
    time_date: str
    has_question: str  # "Y" or "N"
    question_number: str  # Empty string if no question
    forecast_value: str  # JSON string or single value
    error_flag: str  # "Y" or "N"

print("✅ Data structure defined")

✅ Data structure defined


## Utility Functions

In [5]:
def strip_ansi_codes(text: str) -> str:
    """Remove ANSI color codes from text."""
    ansi_escape = re.compile(r'\x1b\[[0-9;]*m')
    return ansi_escape.sub('', text)

print("✅ strip_ansi_codes() defined")

✅ strip_ansi_codes() defined


## GitHub CLI Functions

In [6]:
def verify_gh_cli() -> bool:
    """Verify gh CLI is installed and authenticated."""
    try:
        result = subprocess.run(
            ["gh", "auth", "status"],
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode != 0:
            print("❌ Error: gh CLI is not authenticated. Run 'gh auth login' first.")
            return False
        print("✅ GitHub CLI authenticated")
        return True
    except FileNotFoundError:
        print("❌ Error: gh CLI is not installed. Install from https://cli.github.com/")
        return False
    except Exception as e:
        print(f"❌ Error checking gh CLI: {e}")
        return False

# Test authentication
verify_gh_cli()

✅ GitHub CLI authenticated


True

In [7]:
def get_workflow_runs(repo: str, workflow: str, limit: Optional[int] = None) -> List[Dict]:
    """Fetch workflow runs from GitHub Actions API."""
    print(f"Fetching workflow runs from {repo}...")
    
    cmd = [
        "gh", "api",
        f"repos/{repo}/actions/workflows/{workflow}/runs",
        "--paginate"
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        
        if result.returncode != 0:
            print(f"❌ Command failed: {result.stderr}")
            return []
        
        # Strip ANSI color codes
        clean_output = strip_ansi_codes(result.stdout)
        
        if not clean_output.strip():
            print("❌ Command returned empty output")
            return []
        
        # With --paginate, we get multiple JSON objects separated by newlines
        # Parse each one by counting braces to find complete objects
        runs = []
        json_objects = []
        current_obj = ""
        brace_count = 0
        
        for char in clean_output:
            current_obj += char
            if char == '{':
                brace_count += 1
            elif char == '}':
                brace_count -= 1
                if brace_count == 0 and current_obj.strip():
                    # Complete JSON object found
                    json_objects.append(current_obj.strip())
                    current_obj = ""
        
        print(f"Found {len(json_objects)} paginated responses")
        
        # Parse each JSON object
        for i, json_str in enumerate(json_objects):
            try:
                data = json.loads(json_str)
                if 'workflow_runs' in data:
                    runs.extend(data['workflow_runs'])
                    print(f"  Page {i+1}: {len(data['workflow_runs'])} runs")
            except json.JSONDecodeError as e:
                print(f"  ⚠️  Failed to parse page {i+1}: {e}")
                continue
        
        if not runs:
            print("❌ No runs found in response")
            return []
        
        # Extract just the fields we need
        simplified_runs = []
        for run in runs:
            simplified_runs.append({
                'id': run['id'],
                'run_number': run['run_number'],
                'created_at': run['created_at'],
                'conclusion': run.get('conclusion', '')
            })
        
        # Sort by run_number descending (newest first)
        simplified_runs.sort(key=lambda x: x['run_number'], reverse=True)
        
        # Apply limit if specified
        if limit:
            simplified_runs = simplified_runs[:limit]
        
        print(f"✅ Total: {len(simplified_runs)} runs")
        return simplified_runs
        
    except Exception as e:
        print(f"❌ Error fetching runs: {e}")
        import traceback
        traceback.print_exc()
        return []

# Test: Fetch limited runs
test_runs = get_workflow_runs(REPO, WORKFLOW, limit=5)
if test_runs:
    print("\nSample run:")
    print(f"  Run #{test_runs[0]['run_number']}")
    print(f"  ID: {test_runs[0]['id']}")
    print(f"  Created: {test_runs[0]['created_at']}")
    print(f"  Conclusion: {test_runs[0]['conclusion']}")

Fetching workflow runs from D-Enns/metac-bot-template...
Found 13 paginated responses
  Page 1: 100 runs
  Page 2: 100 runs
  Page 3: 100 runs
  Page 4: 100 runs
  Page 5: 100 runs
  Page 6: 100 runs
  Page 7: 100 runs
  Page 8: 100 runs
  Page 9: 100 runs
  Page 10: 100 runs
  Page 11: 100 runs
  Page 12: 100 runs
  Page 13: 41 runs
✅ Total: 5 runs

Sample run:
  Run #1241
  ID: 21834867844
  Created: 2026-02-09T17:40:57Z
  Conclusion: success


In [8]:
def download_run_log(repo: str, run_id: int, run_number: int) -> Optional[str]:
    """Download log for a single workflow run."""
    cmd = ["gh", "run", "view", str(run_id), "--repo", repo, "--log"]
    
    try:
        # Capture as bytes to avoid subprocess encoding issues
        result = subprocess.run(
            cmd, 
            capture_output=True,
            timeout=120
        )
        
        if result.returncode != 0:
            print(f"  ❌ Failed to download run {run_number}")
            return None
        
        # Manually decode with error handling
        try:
            log_text = result.stdout.decode('utf-8', errors='replace')
        except Exception as e:
            print(f"  ⚠️  UTF-8 decode failed, trying latin-1: {e}")
            # Fallback: latin-1 accepts all byte values
            log_text = result.stdout.decode('latin-1', errors='replace')
        
        # Strip ANSI codes from log output
        return strip_ansi_codes(log_text)
        
    except Exception as e:
        print(f"  ❌ Error downloading run {run_number}: {e}")
        return None

print("✅ Download function defined (robust byte handling)")

✅ Download function defined (robust byte handling)


## Test: Download and Inspect Single Log

Let's download one log to see what we're working with.

In [9]:
# Get the most recent run
if test_runs:
    sample_run = test_runs[0]
    print(f"Downloading log for run #{sample_run['run_number']}...")
    sample_log = download_run_log(REPO, sample_run['id'], sample_run['run_number'])
    
    if sample_log:
        print(f"✅ Downloaded log ({len(sample_log)} characters)")
        print(f"\nFirst 1000 characters:")
        print("=" * 80)
        print(sample_log[:1000])
        print("=" * 80)
    else:
        print("❌ Failed to download log")

✅ Downloaded log (80582 characters)

First 1000 characters:
forecast_job	Set up job	﻿2026-02-09T17:41:01.9193431Z Current runner version: '2.331.0'
forecast_job	Set up job	2026-02-09T17:41:01.9230117Z ##[group]Runner Image Provisioner
forecast_job	Set up job	2026-02-09T17:41:01.9231506Z Hosted Compute Agent
forecast_job	Set up job	2026-02-09T17:41:01.9232520Z Version: 20260123.484
forecast_job	Set up job	2026-02-09T17:41:01.9233588Z Commit: 6bd6555ca37d84114959e1c76d2c01448ff61c5d
forecast_job	Set up job	2026-02-09T17:41:01.9235166Z Build Date: 2026-01-23T19:41:17Z
forecast_job	Set up job	2026-02-09T17:41:01.9236326Z Worker ID: {0c8273f8-36d6-4b68-93bd-2dbf0b3c49cd}
forecast_job	Set up job	2026-02-09T17:41:01.9237591Z Azure Region: centralus
forecast_job	Set up job	2026-02-09T17:41:01.9238610Z ##[endgroup]
forecast_job	Set up job	2026-02-09T17:41:01.9240975Z ##[group]Operating System
forecast_job	Set up job	2026-02-09T17:41:01.9242058Z Ubuntu
forecast_job	Set up job	2026-02-09T17:41:01

## Data Extraction Functions

Now let's build the functions to extract each piece of information from the logs.

In [10]:
# Regex patterns (reused from existing tools)
TIMESTAMP_PATTERN = re.compile(r'(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z)')
QUESTION_URL_PATTERN = re.compile(r'https://www\.metaculus\.com/questions/(\d+)')
FOUND_RESEARCH_PATTERN = re.compile(r'Found Research for URL\s+(https://www\.metaculus\.com/questions/\d+)')
ERROR_EXIT_PATTERN = re.compile(r'Error: Process completed with exit code 1\.')

print("✅ Regex patterns defined")

✅ Regex patterns defined


In [11]:
def extract_timestamp(log_content: str) -> str:
    """Extract first Python logger timestamp from log."""
    # Look for Python logger format: YYYY-MM-DD HH:MM:SS (space-separated, not ISO format)
    # Pattern: YYYY-MM-DD HH:MM:SS followed by comma and milliseconds
    pattern = re.compile(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})')
    
    match = pattern.search(log_content)
    if match:
        return match.group(1)
    
    return ""  # Fallback if not found

print("✅ extract_timestamp() defined")

✅ extract_timestamp() defined


In [12]:
def check_has_question(log_content: str) -> str:
    """Check if run processed a question (Y/N)."""
    if FOUND_RESEARCH_PATTERN.search(log_content):
        return "Y"
    return "N"

print("✅ check_has_question() defined")

✅ check_has_question() defined


In [13]:
def extract_question_number(log_content: str) -> str:
    """Extract question number from Metaculus URL."""
    match = QUESTION_URL_PATTERN.search(log_content)
    return match.group(1) if match else ""

print("✅ extract_question_number() defined")

✅ extract_question_number() defined


In [14]:
def detect_question_type(log_content: str) -> Optional[str]:
    """Detect question type from log content."""
    if "BinaryQuestion" in log_content or "*Final Prediction*:" in log_content:
        return "Binary"
    elif "MultipleChoiceQuestion" in log_content:
        return "MultipleChoice"
    elif "NumericQuestion" in log_content or "Probability distribution:" in log_content:
        return "Numeric"
    return None

print("✅ detect_question_type() defined")

✅ detect_question_type() defined


In [15]:
def extract_binary_forecast(log_content: str) -> str:
    """Extract binary forecast percentage."""
    patterns = [
        r'### Final Prediction[^\n]*\n[^\n]*?(\d+\.?\d*)%?',  # Three hashes (in logs)
        r'\*Final Prediction\*:\s*(\d+\.?\d*)%?',             # Markdown format
        r'\*\*Probability:\s*(\d+\.?\d*)%?'                   # Alternative format
    ]
    
    for pattern in patterns:
        match = re.search(pattern, log_content)
        if match:
            return match.group(1)
    
    raise ValueError("Binary forecast not found")

print("✅ extract_binary_forecast() defined")

✅ extract_binary_forecast() defined


In [16]:
def extract_mc_forecast(log_content: str) -> str:
    """Extract multiple choice forecast as JSON."""
    # Look for "### Final Answer" or "# Final Answer" section with JSON dict
    patterns = [
        r'### Final Answer[^\n]*\n\s*(\{[^}]+\})',  # Three hashes (in logs)
        r'# Final Answer[^\n]*\n\s*(\{[^}]+\})',   # One hash (in summaries)
    ]
    
    for pattern in patterns:
        section_match = re.search(pattern, log_content)
        if section_match:
            return section_match.group(1)
    
    # Alternative: Parse bullet list format
    # - Option A: 60.5%
    # - Option B: 39.5%
    pattern = r'-\s*([^:]+):\s*(\d+\.?\d*)%?'
    matches = re.findall(pattern, log_content)
    
    if matches:
        forecast_dict = {opt.strip(): float(pct) for opt, pct in matches}
        return json.dumps(forecast_dict)
    
    raise ValueError("Multiple choice forecast not found")

print("✅ extract_mc_forecast() defined")

✅ extract_mc_forecast() defined


In [17]:
def extract_numeric_forecast(log_content: str) -> str:
    """Extract numeric forecast list as JSON."""
    # Look for "### Final Answer" (three hashes in logs) followed by array
    # Try different heading variations
    patterns = [
        r'### Final Answer[^\n]*\n\s*(\[[\d.,\s]+\])',  # Three hashes (in logs)
        r'# Final Answer[^\n]*\n\s*(\[[\d.,\s]+\])',   # One hash (in summaries)
    ]
    
    for pattern in patterns:
        section_match = re.search(pattern, log_content)
        if section_match:
            return section_match.group(1)
    
    raise ValueError("Numeric forecast list not found")

print("✅ extract_numeric_forecast() defined")

✅ extract_numeric_forecast() defined


In [18]:
def extract_forecast_value(log_content: str, question_type: Optional[str]) -> str:
    """Extract forecast value based on question type."""
    if not question_type:
        return ""
    
    try:
        if question_type == "Binary":
            return extract_binary_forecast(log_content)
        elif question_type == "MultipleChoice":
            return extract_mc_forecast(log_content)
        elif question_type == "Numeric":
            return extract_numeric_forecast(log_content)
    except Exception as e:
        return f"ERROR: {str(e)}"
    
    return ""

print("✅ extract_forecast_value() defined")

✅ extract_forecast_value() defined


In [19]:
def check_error_flag(log_content: str) -> str:
    """Check if run had error exit code (Y/N)."""
    # Check last 200 lines for error pattern
    last_lines = '\n'.join(log_content.split('\n')[-200:])
    
    if ERROR_EXIT_PATTERN.search(last_lines):
        return "Y"
    return "N"

print("✅ check_error_flag() defined")

✅ check_error_flag() defined


In [20]:
def process_log(log_content: str, run_number: int) -> RunInfo:
    """Process a single log file and extract all fields."""
    
    # Extract timestamp
    time_date = extract_timestamp(log_content)
    
    # Check for question
    has_question = check_has_question(log_content)
    
    # Extract question number (if present)
    question_number = extract_question_number(log_content) if has_question == "Y" else ""
    
    # Extract forecast value (if question present)
    forecast_value = ""
    if has_question == "Y":
        question_type = detect_question_type(log_content)
        forecast_value = extract_forecast_value(log_content, question_type)
    
    # Check error flag
    error_flag = check_error_flag(log_content)
    
    return RunInfo(
        workflow_run_number=run_number,
        time_date=time_date,
        has_question=has_question,
        question_number=question_number,
        forecast_value=forecast_value,
        error_flag=error_flag
    )

print("✅ process_log() defined")

✅ process_log() defined


## Test: Extract Data from Sample Log

Let's test our extraction functions on the sample log we downloaded.

In [21]:
if sample_log:
    print(f"Testing extraction on run #{sample_run['run_number']}...\n")
    
    # Test each extraction function
    print(f"Timestamp: {extract_timestamp(sample_log)}")
    print(f"Has question: {check_has_question(sample_log)}")
    print(f"Question number: {extract_question_number(sample_log)}")
    print(f"Question type: {detect_question_type(sample_log)}")
    print(f"Error flag: {check_error_flag(sample_log)}")
    
    # Test full processing
    print("\n" + "=" * 80)
    print("Full RunInfo extraction:")
    print("=" * 80)
    run_info = process_log(sample_log, sample_run['run_number'])
    print(f"Run Number: {run_info.workflow_run_number}")
    print(f"Time/Date: {run_info.time_date}")
    print(f"Has Question: {run_info.has_question}")
    print(f"Question Number: {run_info.question_number}")
    print(f"Forecast Value: {run_info.forecast_value}")
    print(f"Error Flag: {run_info.error_flag}")
else:
    print("⚠️  No sample log available for testing")

Testing extraction on run #1241...

Timestamp: 2026-02-09 17:41:47
Has question: N
Question number: 
Question type: None
Error flag: N

Full RunInfo extraction:
Run Number: 1241
Time/Date: 2026-02-09 17:41:47
Has Question: N
Question Number: 
Forecast Value: 
Error Flag: N


## Process Multiple Runs

Now let's process multiple runs to build our dataset.

In [22]:
def process_runs(repo: str, workflow: str, limit: Optional[int] = None) -> List[RunInfo]:
    """Process multiple workflow runs and extract data."""
    
    # Get runs
    runs = get_workflow_runs(repo, workflow, limit=limit)
    
    if not runs:
        print("No runs found")
        return []
    
    print(f"\nProcessing {len(runs)} runs...\n")
    
    run_info_list = []
    
    for i, run in enumerate(runs, 1):
        run_id = run['id']
        run_number = run['run_number']
        
        print(f"[{i}/{len(runs)}] Processing run #{run_number}...", end='')
        
        # Download log
        log_content = download_run_log(repo, run_id, run_number)
        
        if not log_content:
            print(" ❌ Failed to download")
            continue
        
        # Extract info
        try:
            run_info = process_log(log_content, run_number)
            run_info_list.append(run_info)
            print(f" ✓ (Question: {run_info.has_question}, Error: {run_info.error_flag})")
        except Exception as e:
            print(f" ⚠️  Error: {e}")
    
    print(f"\n✅ Processed {len(run_info_list)} runs successfully")
    return run_info_list

print("✅ process_runs() defined")

✅ process_runs() defined


In [23]:
# Process test batch
print(f"Processing {TEST_LIMIT} runs for testing...\n")
test_data = process_runs(REPO, WORKFLOW, limit=TEST_LIMIT)

Processing 10 runs for testing...

Fetching workflow runs from D-Enns/metac-bot-template...
Found 13 paginated responses
  Page 1: 100 runs
  Page 2: 100 runs
  Page 3: 100 runs
  Page 4: 100 runs
  Page 5: 100 runs
  Page 6: 100 runs
  Page 7: 100 runs
  Page 8: 100 runs
  Page 9: 100 runs
  Page 10: 100 runs
  Page 11: 100 runs
  Page 12: 100 runs
  Page 13: 41 runs
✅ Total: 10 runs

Processing 10 runs...

 ✓ (Question: N, Error: N)1...
 ✓ (Question: N, Error: N)0...
 ✓ (Question: N, Error: N)9...
 ✓ (Question: N, Error: N)8...
 ✓ (Question: Y, Error: N)7...
 ✓ (Question: N, Error: N)6...
 ✓ (Question: N, Error: N)5...
 ✓ (Question: N, Error: N)4...
 ✓ (Question: N, Error: N)3...
 ✓ (Question: N, Error: N)32...

✅ Processed 10 runs successfully


In [24]:
# Debug: Inspect a log with a question to see actual format
# Find a run with a question from test_data
run_with_question = None
for info in test_data:
    if info.has_question == "Y":
        run_with_question = info
        break

if run_with_question:
    # Re-download the log for inspection
    run_num = run_with_question.workflow_run_number
    run_id = None
    for run in test_runs:
        if run['run_number'] == run_num:
            run_id = run['id']
            break
    
    if run_id:
        print(f"Inspecting log for run #{run_num} (has question)...")
        log = download_run_log(REPO, run_id, run_num)
        
        if log:
            print(f"\n{'='*80}")
            print("SEARCHING FOR TIMESTAMP...")
            print(f"{'='*80}")
            # Show first 100 lines to find timestamp format
            lines = log.split('\n')[:100]
            for i, line in enumerate(lines):
                if 'python main.py' in line.lower() or '2026-02-' in line:
                    print(f"Line {i}: {line[:150]}")
            
            print(f"\n{'='*80}")
            print("SEARCHING FOR FORECAST VALUE...")
            print(f"{'='*80}")
            # Search for forecast patterns
            forecast_section = ""
            in_forecast = False
            for line in log.split('\n'):
                if 'Final Answer' in line or 'Final Prediction' in line:
                    in_forecast = True
                    forecast_section = line + '\n'
                elif in_forecast:
                    forecast_section += line + '\n'
                    if len(forecast_section) > 500:  # Capture ~10 lines
                        break
            
            if forecast_section:
                print(forecast_section[:1000])
            else:
                print("⚠️  'Final Answer' or 'Final Prediction' not found in log")
                print("\nSearching for bracket patterns [...]:")
                for i, line in enumerate(log.split('\n')):
                    if '[' in line and ']' in line and any(char.isdigit() for char in line):
                        print(f"Line {i}: {line[:150]}")
                        if i > 50:  # Show first 50 matches
                            break
else:
    print("No runs with questions found in test_data")

Inspecting log for run #1237 (has question)...

SEARCHING FOR TIMESTAMP...
Line 0: forecast_job	UNKNOWN STEP	﻿2026-02-09T13:48:49.1306136Z Current runner version: '2.331.0'
Line 1: forecast_job	UNKNOWN STEP	2026-02-09T13:48:49.1339272Z ##[group]Runner Image Provisioner
Line 2: forecast_job	UNKNOWN STEP	2026-02-09T13:48:49.1340471Z Hosted Compute Agent
Line 3: forecast_job	UNKNOWN STEP	2026-02-09T13:48:49.1341415Z Version: 20260123.484
Line 4: forecast_job	UNKNOWN STEP	2026-02-09T13:48:49.1342355Z Commit: 6bd6555ca37d84114959e1c76d2c01448ff61c5d
Line 5: forecast_job	UNKNOWN STEP	2026-02-09T13:48:49.1343600Z Build Date: 2026-01-23T19:41:17Z
Line 6: forecast_job	UNKNOWN STEP	2026-02-09T13:48:49.1344736Z Worker ID: {9a701ee3-39cb-4d5e-accd-2e8e4a3d8fc0}
Line 7: forecast_job	UNKNOWN STEP	2026-02-09T13:48:49.1345794Z Azure Region: eastus2
Line 8: forecast_job	UNKNOWN STEP	2026-02-09T13:48:49.1346738Z ##[endgroup]
Line 9: forecast_job	UNKNOWN STEP	2026-02-09T13:48:49.1349546Z ##[group]Operati

In [25]:
# Debug: Test forecast extraction on actual log
if test_data:
    # Find a run with a question
    for info in test_data:
        if info.has_question == "Y":
            run_num = info.workflow_run_number
            
            # Re-download log
            run_id = None
            for run in test_runs:
                if run['run_number'] == run_num:
                    run_id = run['id']
                    break
            
            if run_id:
                print(f"Testing forecast extraction on run #{run_num}...")
                log = download_run_log(REPO, run_id, run_num)
                
                if log:
                    # Test question type detection
                    q_type = detect_question_type(log)
                    print(f"Detected question type: {q_type}")
                    
                    # Try extracting forecast
                    print("\nAttempting forecast extraction...")
                    try:
                        forecast = extract_forecast_value(log, q_type)
                        print(f"✅ Extracted forecast: {forecast}")
                    except Exception as e:
                        print(f"❌ Extraction failed: {e}")
                        import traceback
                        traceback.print_exc()
                    
                    # Show what we're looking for
                    print("\n" + "="*80)
                    print("Searching for '### Final Answer' in log...")
                    print("="*80)
                    for i, line in enumerate(log.split('\n')):
                        if 'Final Answer' in line or 'Final Prediction' in line:
                            # Show context around the match
                            lines = log.split('\n')
                            start = max(0, i-1)
                            end = min(len(lines), i+5)
                            print(f"\nFound at line {i}:")
                            for j in range(start, end):
                                print(f"  Line {j}: {lines[j][:100]}")
                            break
                break

Testing forecast extraction on run #1237...
Detected question type: None

Attempting forecast extraction...
✅ Extracted forecast: 

Searching for '### Final Answer' in log...

Found at line 933:
  Line 932: forecast_job	UNKNOWN STEP	2026-02-09T13:50:33.2092602Z 
  Line 933: forecast_job	UNKNOWN STEP	2026-02-09T13:50:33.2092689Z ### Final Answer (Points)
  Line 934: forecast_job	UNKNOWN STEP	2026-02-09T13:50:33.2092941Z [83.5, 85.0, 86.0, 86.5, 88.0, 89.0, 89.5, 91
  Line 935: forecast_job	UNKNOWN STEP	2026-02-09T13:50:33.2093599Z 2026-02-09 13:50:33,205 - forecasting_tools.a
  Line 936: forecast_job	UNKNOWN STEP	2026-02-09T13:50:33.2094664Z 2026-02-09 13:50:33,207 - forecasting_tools.a
  Line 937: forecast_job	UNKNOWN STEP	2026-02-09T13:50:33.3531640Z 2026-02-09 13:50:33,352 - main - INFO - Reaso


In [26]:
def write_to_excel(run_info_list: List[RunInfo], output_file: Path):
    """Write extracted data to Excel file with versioning."""
    
    # Find next version number
    base_name = output_file.stem  # "Runs and Question Numbers"
    extension = output_file.suffix  # ".xlsx"
    directory = output_file.parent
    
    version = 1
    while True:
        versioned_name = f"{base_name}_v{version:03d}{extension}"
        versioned_path = directory / versioned_name
        if not versioned_path.exists():
            break
        version += 1
    
    print(f"Creating new workbook: {versioned_path}")
    
    # Always create fresh workbook (no loading existing)
    wb = Workbook()
    ws = wb.active
    
    # Create header row
    headers = [
        'workflow_run_number',
        'time_date',
        'has_question',
        'question_number',
        'forecast_value',
        'error_flag'
    ]
    ws.append(headers)
    
    # Format header row
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')
    
    # Sort by run_number descending
    sorted_runs = sorted(run_info_list, key=lambda x: x.workflow_run_number, reverse=True)
    
    # Append data rows (one per run)
    for run_info in sorted_runs:
        ws.append([
            run_info.workflow_run_number,
            run_info.time_date,
            run_info.has_question,
            run_info.question_number,
            run_info.forecast_value,
            run_info.error_flag
        ])
    
    # Set column widths
    ws.column_dimensions['A'].width = 20  # workflow_run_number
    ws.column_dimensions['B'].width = 20  # time_date
    ws.column_dimensions['C'].width = 15  # has_question
    ws.column_dimensions['D'].width = 18  # question_number
    ws.column_dimensions['E'].width = 50  # forecast_value
    ws.column_dimensions['F'].width = 12  # error_flag
    
    # Ensure parent directory exists
    directory.mkdir(parents=True, exist_ok=True)
    
    # Save
    wb.save(versioned_path)
    print(f"✅ Excel file saved: {versioned_path}")
    print(f"   Total rows: {len(sorted_runs)} (plus header)")
    return versioned_path

print("✅ write_to_excel() defined (with versioning)")

✅ write_to_excel() defined (with versioning)


# Write test data to Excel (with versioning)
if test_data:
    versioned_file = write_to_excel(test_data, OUTPUT_FILE)
    print(f"\n✅ Created: {versioned_file.name}")
else:
    print("⚠️  No data to write")

In [27]:
def write_to_excel(run_info_list: List[RunInfo], output_file: Path):
    """Write extracted data to Excel file."""
    
    # Create or load workbook
    if output_file.exists():
        print(f"Loading existing workbook: {output_file}")
        wb = load_workbook(output_file)
        ws = wb.active
    else:
        print(f"Creating new workbook: {output_file}")
        wb = Workbook()
        ws = wb.active
        
        # Create header row
        headers = [
            'workflow_run_number',
            'time_date',
            'has_question',
            'question_number',
            'forecast_value',
            'error_flag'
        ]
        ws.append(headers)
        
        # Format header row
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(horizontal='center')
    
    # Sort by run_number descending
    sorted_runs = sorted(run_info_list, key=lambda x: x.workflow_run_number, reverse=True)
    
    # Append data rows
    for run_info in sorted_runs:
        ws.append([
            run_info.workflow_run_number,
            run_info.time_date,
            run_info.has_question,
            run_info.question_number,
            run_info.forecast_value,
            run_info.error_flag
        ])
    
    # Set column widths
    ws.column_dimensions['A'].width = 20  # workflow_run_number
    ws.column_dimensions['B'].width = 20  # time_date
    ws.column_dimensions['C'].width = 15  # has_question
    ws.column_dimensions['D'].width = 18  # question_number
    ws.column_dimensions['E'].width = 50  # forecast_value
    ws.column_dimensions['F'].width = 12  # error_flag
    
    # Ensure parent directory exists
    output_file.parent.mkdir(parents=True, exist_ok=True)
    
    # Save
    wb.save(output_file)
    print(f"✅ Excel file saved: {output_file}")
    print(f"   Total rows: {len(sorted_runs)} (plus header)")

print("✅ write_to_excel() defined")

✅ write_to_excel() defined


In [28]:
# Inspect Excel file contents
if OUTPUT_FILE.exists():
    wb = load_workbook(OUTPUT_FILE)
    ws = wb.active
    
    print("Excel File Contents:")
    print("=" * 100)
    
    # Read all rows
    rows = list(ws.iter_rows(values_only=True))
    
    # Print header
    print(f"{'Run #':<10} {'Time/Date':<20} {'Has Q':<8} {'Q #':<10} {'Forecast Value':<40} {'Error':<8}")
    print("-" * 100)
    
    # Print data rows (skip header)
    for row in rows[1:]:
        run_num, time_date, has_q, q_num, forecast, error = row
        forecast_str = str(forecast)[:40] if forecast else ""
        print(f"{run_num:<10} {time_date or '':<20} {has_q:<8} {q_num or '':<10} {forecast_str:<40} {error:<8}")
    
    print("=" * 100)
    print(f"\nTotal rows: {len(rows) - 1}")
    
    # Analyze issues
    rows_with_questions = sum(1 for row in rows[1:] if row[2] == "Y")
    rows_with_timestamps = sum(1 for row in rows[1:] if row[1])
    rows_with_forecast = sum(1 for row in rows[1:] if row[4])
    
    print(f"\nRows with questions: {rows_with_questions}")
    print(f"Rows with timestamps: {rows_with_timestamps}")
    print(f"Rows with forecast values: {rows_with_forecast}")
else:
    print("Excel file not found")

Excel File Contents:
Run #      Time/Date            Has Q    Q #        Forecast Value                           Error   
----------------------------------------------------------------------------------------------------
1239                            N                                                            N       
1238                            N                                                            N       
1237                            Y        42040                                               N       
1236                            N                                                            N       
1235                            N                                                            N       
1234                            N                                                            N       
1233                            N                                                            N       
1232                            N                             

In [29]:
# Write test data to Excel
if test_data:
    write_to_excel(test_data, OUTPUT_FILE)
else:
    print("⚠️  No data to write")

Loading existing workbook: C:\Users\Donni\projects\metac_bot_Spring_2026\products\Runs and Question Numbers.xlsx
✅ Excel file saved: C:\Users\Donni\projects\metac_bot_Spring_2026\products\Runs and Question Numbers.xlsx
   Total rows: 10 (plus header)


## Full Processing

Once the test looks good, run this cell to process ALL runs (no limit).

**⚠️ Warning:** Processing all 1224 runs will take 40-60 minutes and download ~600MB of log data.

In [30]:
# UNCOMMENT TO PROCESS ALL RUNS (this will take a while!)
# print("⚠️  Processing ALL runs - this may take 40-60 minutes...\n")
# full_data = process_runs(REPO, WORKFLOW, limit=None)
# if full_data:
#     write_to_excel(full_data, OUTPUT_FILE)
#     print(f"\n🎉 Complete! Processed {len(full_data)} runs")

---

## Summary Statistics

Optional: Analyze the extracted data.

In [31]:
if test_data:
    total_runs = len(test_data)
    runs_with_questions = sum(1 for r in test_data if r.has_question == "Y")
    runs_with_errors = sum(1 for r in test_data if r.error_flag == "Y")
    
    print("=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)
    print(f"Total runs processed: {total_runs}")
    print(f"Runs with questions: {runs_with_questions} ({runs_with_questions/total_runs*100:.1f}%)")
    print(f"Runs with errors: {runs_with_errors} ({runs_with_errors/total_runs*100:.1f}%)")
    print(f"Success rate: {(total_runs - runs_with_errors)/total_runs*100:.1f}%")
    print("=" * 80)

SUMMARY STATISTICS
Total runs processed: 10
Runs with questions: 1 (10.0%)
Runs with errors: 0 (0.0%)
Success rate: 100.0%
